# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row = one content item (page), for one client, on one day — from fact_content_daily_performance, grain is client_hash_id × content_hash_id × report_date. Time window: I'm iterating on a mid-panel month, month=2026-03, per the assignment's warning to keep the final month sealed as a test month.

In [17]:
import os, getpass, duckdb, pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}
print("Connected.")

Connected.


## 2. Fields: feature / label / context / excluded

Features: gsc_impressions, gsc_clicks, gsc_avg_position, days_since_last_update (or equivalent from dim_content), query-mix signals from the query table (visible_queries, rare_share).
Label: derived from trend_direction-style logic impressions declining >20% month-over-month between two 30-day windows.
Context (joins/grouping only, never features): client_hash_id, content_hash_id, report_date.
Excluded: trend_pct and any pre-computed decline/health flags excluded because they're either the label itself in disguise (leakage) or a product-team decision output, not raw observable data. Also excluding ga4_data_available == NULL rows from any GA4-dependent feature, since that flag can be genuinely missing, not just false.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

These three queries confirm: (1) grain is truly one row per client-content-day, (2) March 2026 spans the expected ~31 days across N clients with M total rows, (3) filtering ga4_data_available IS TRUE (not = TRUE) correctly excludes both FALSE and NULL rows the dictionary warns NULL is a real third state, not just missing.

In [11]:
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) as n
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
""").df()
print("Rows violating 1-row-per-client-content-day grain:", len(grain_check))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating 1-row-per-client-content-day grain: 0


In [12]:
span = con.sql(f"""
    SELECT COUNT(*) as n_rows, MIN(report_date) as first_day, MAX(report_date) as last_day,
           COUNT(DISTINCT client_hash_id) as n_clients
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""").df()
print(span)

    n_rows  first_day   last_day  n_clients
0  9841378 2026-03-01 2026-03-31         55


In [13]:
avail = con.sql(f"""
    SELECT COUNT(*) as total_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as ga4_available_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""").df()
print(avail)
print(f"Share with GA4 available: {avail['ga4_available_rows'][0] / avail['total_rows'][0]:.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  ga4_available_rows
0     9841378            413966.0
Share with GA4 available: 0.042


In [14]:
features_march = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) as impressions_30d,
           SUM(gsc_clicks) as clicks_30d,
           AVG(gsc_avg_position) as avg_position_30d,
           STDDEV(gsc_avg_position) as position_volatility_30d,
           COUNT(DISTINCT report_date) as days_with_data
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
    GROUP BY 1,2
""").df()
features_march.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions_30d,clicks_30d,avg_position_30d,position_volatility_30d,days_with_data
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,77.0,0.0,4.074107,2.678448,31
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,602.0,4.0,4.428747,1.809163,31
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,810.0,1.0,4.866123,1.853147,31
3,client_62f4a7e64f5e0096,content_ceaec531566ffcfc,82.0,0.0,8.978086,9.774390,31
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,1858.0,6.0,1.854929,0.691448,31



*    impressions_30d — knowable at decision moment because it's a completed

*    clicks_30d — same reasoning, trailing sum only.

*   avg_position_30d — average of past daily positions only.

*   position_volatility_30d — standard deviation of past positions, no future leakage.

*   days_with_data — count of past days observed, purely historical.





In [15]:
import numpy as np

# Build a quick label + honest score
label_month = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) as impressions_next
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-04-01' AND report_date < '2026-05-01'
    GROUP BY 1,2
""").df()

merged = features_march.merge(label_month, on=['client_hash_id','content_hash_id'], how='inner')
merged['is_declining'] = (merged['impressions_next'] < 0.8 * merged['impressions_30d']).astype(int)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_cols = ['impressions_30d','clicks_30d','avg_position_30d','position_volatility_30d','days_with_data']
X = merged[honest_cols].fillna(0)
y = merged['is_declining']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42)
m = RandomForestClassifier(random_state=42).fit(Xtr, ytr)
print("Honest AUC:", roc_auc_score(yte, m.predict_proba(Xte)[:,1]))

# --- Now deliberately add a label-derived leaked column ---
merged['leaked_pct_change'] = (merged['impressions_next'] - merged['impressions_30d']) / merged['impressions_30d']
merged['leaked_pct_change'] = merged['leaked_pct_change'].replace([np.inf, -np.inf], np.nan)

X_leak = merged[honest_cols + ['leaked_pct_change']].replace([np.inf, -np.inf], np.nan).fillna(0)
Xtr2, Xte2, ytr2, yte2 = train_test_split(X_leak, y, test_size=0.25, random_state=42)
m2 = RandomForestClassifier(random_state=42).fit(Xtr2, ytr2)
print("Leaked AUC:", roc_auc_score(yte2, m2.predict_proba(Xte2)[:,1]), "<- jumps toward 1.0, because this column IS the label")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Honest AUC: 0.8703070854752516
Leaked AUC: 1.0 <- jumps toward 1.0, because this column IS the label


Adding leaked_pct_change which is directly derived from the future value used to build the label pushed AUC toward perfect. That's the tell: a feature is leaking when it's mathematically entangled with the label itself. I've kept only the honest five-feature set going forward.

## 4. Data limits

This slice can never tell me: (1) why a page declined no causal signal, only correlational; (2) whether patterns hold for clients with short history per dim_clients, history depth is unbalanced (some clients have 17 months, some 3), and March 2026 will include some clients recently onboarded with thin history; (3) anything about pages with ga4_data_available NULL, since that's neither confirmed available nor confirmed unavailable I exclude those rather than guess. One named limitation: this month (March 2026) only reflects behavior for clients active at that time client churn or onboarding mid-panel means the slice isn't a stable, representative sample across the full 17-month history.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.